# Task 1B — Constrained Run 1 : Détection d'enthymèmes (3 classes)

**Entrée** : texte brut du tweet uniquement (aucune information d'annotation autorisée)  
**Sortie** : label parmi `premise`, `conclusion`, `none` + vecteur de probabilités  

Ce notebook compare deux stratégies d'adaptation :
- **Full Fine-Tuning** : mise à jour de tous les paramètres du modèle
- **LoRA** : adaptation par matrices de bas rang (PEFT), plus robuste sur petits datasets

Métrique principale : **Macro F1** (mode 2 classes fusionnées = métrique de classement officielle)  
Métrique secondaire : Macro F1 en mode 3 classes + Cross-Entropy sur soft labels

## 0. Installation des dépendances

In [1]:
# Installer les bibliothèques nécessaires
# peft  : Parameter-Efficient Fine-Tuning (LoRA)
# transformers / datasets / evaluate : écosystème HuggingFace
# accelerate : backend d'entraînement multi-GPU/CPU
# scikit-learn : métriques supplémentaires
!pip install transformers datasets evaluate peft accelerate scikit-learn -q


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 1. Imports et configuration globale

Charge le fichier JSON du jeu de données et construit trois objets pour chaque tweet :
- hard_label : l'étiquette majoritaire (0/1/2), utilisée comme cible principale
- soft_label : la distribution des votes [P(premise), P(conclusion), P(none)], enregistrée mais non utilisée lors de l'entraînement du Run 1 — elle sert uniquement à calculer l'entropie croisée d'évaluation requise par l'organisateur
- disagreement_score : non utilisé dans le Run 1, préparé pour de futures comparaisons

Gère également le cas où le JSON ne comporte pas de champ split explicite, en effectuant une découpe automatique 90/10.

In [ ]:
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from sklearn.metrics import classification_report, f1_score

# ── Reproductibilité ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Device utilisé : cuda


In [ ]:
# ── Chemins vers les données ──────────────────────────────────────────────────
# Adapter ces chemins selon l'organisation locale du projet
#DATA_DIR   = Path("data")
TRAIN_FILE = "/data-lachesis/common/propp/DeniseAtzori/notebooks/DeniseAtzori/LoRA/merged_annotations_public.json"

# Le fichier JSON contient train + dev ; on les sépare via un champ 'split'
# (si absent, on fait un split 90/10 manuellement)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Paramètres du modèle ──────────────────────────────────────────────────────
# DeBERTa-v3-base est un bon compromis taille/performance pour ce dataset modeste
# Alternatives : "roberta-large", "microsoft/deberta-v3-large"
MODEL_NAME  = "microsoft/deberta-v3-base" # 128 M paramétres
MAX_LENGTH  = 128   # les tweets sont courts, 128 tokens suffisent largement
NUM_LABELS  = 3

# ── Correspondance label ↔ indice ────────────────────────────────────────────
LABEL2ID = {"premise": 0, "conclusion": 1, "none": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé : {DEVICE}")

Device utilisé : cuda


## 2. Chargement et préparation des données

In [4]:
def load_dataset_json(filepath: Path):
    """
    Charge le fichier JSON annoté et retourne deux listes :
      - train_samples : liste de dicts {text, hard_label, soft_label}
      - dev_samples   : idem

    Structure JSON attendue par entrée :
    {
      "id": "1",
      "tweet_text": "...",
      "source": "vaccine",
      "annotations": [
        {"label": "premise",    "implicit_text": "..."},
        {"label": "premise",    "implicit_text": "..."},
        {"label": "none",       "implicit_text": ""}
      ],
      "majority_label": "premise",
      "split": "train"   ← champ optionnel
    }
    """
    with open(filepath, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # Si le champ 'split' est absent, on fait une séparation 90/10 aléatoire
    has_split_field = "split" in raw[0]
    if not has_split_field:
        random.shuffle(raw)
        cut = int(len(raw) * 0.9)
        for i, entry in enumerate(raw):
            entry["split"] = "train" if i < cut else "dev"

    def build_soft_label(annotations):
        """
        Construit un vecteur de probabilités [P(premise), P(conclusion), P(none)]
        à partir des votes bruts des annotateurs.
        Exemple : [premise, premise, none] → [0.67, 0.0, 0.33]

        Note : pour Constrained Run 1, ce soft_label n'est PAS utilisé à
        l'entraînement (on ne peut pas utiliser les annotations).
        Il est stocké ici pour évaluation uniquement.
        """
        counts = np.zeros(NUM_LABELS)
        for ann in annotations:
            idx = LABEL2ID.get(ann["label"], 2)  # 'none' par défaut si label inconnu
            counts[idx] += 1
        return (counts / counts.sum()).tolist()

    train_samples, dev_samples = [], []
    for entry in raw:
        sample = {
            "id"         : entry["id"],
            "text"       : entry["tweet_text"],
            "hard_label" : LABEL2ID[entry["majority_label"]],
            "soft_label" : build_soft_label(entry["annotations"]),
        }
        if entry["split"] == "train":
            train_samples.append(sample)
        else:
            dev_samples.append(sample)

    print(f"Train : {len(train_samples)} exemples")
    print(f"Dev   : {len(dev_samples)} exemples")
    return train_samples, dev_samples


train_samples, dev_samples = load_dataset_json(TRAIN_FILE)

Train : 1199 exemples
Dev   : 134 exemples


In [5]:
# ── Vérification de la distribution des labels ────────────────────────────────
def label_distribution(samples, split_name):
    from collections import Counter
    counts = Counter(s["hard_label"] for s in samples)
    total  = sum(counts.values())
    print(f"\n{split_name} — distribution des labels :")
    for idx, name in ID2LABEL.items():
        n = counts.get(idx, 0)
        print(f"  {name:12s} : {n:4d}  ({100*n/total:.1f}%)")

label_distribution(train_samples, "Train")
label_distribution(dev_samples,   "Dev")


Train — distribution des labels :
  premise      :  326  (27.2%)
  conclusion   :   81  (6.8%)
  none         :  792  (66.1%)

Dev — distribution des labels :
  premise      :   38  (28.4%)
  conclusion   :   11  (8.2%)
  none         :   85  (63.4%)


In [6]:
# ── Calcul des poids de classe pour compenser le déséquilibre ────────────────
# Le dataset est fortement déséquilibré : ~66% 'none', ~27% 'premise', ~7% 'conclusion'
# On pondère la loss en inversant les fréquences (stratégie classique)

from collections import Counter

def compute_class_weights(samples):
    counts = Counter(s["hard_label"] for s in samples)
    total  = sum(counts.values())
    # Poids inversement proportionnel à la fréquence, normalisé
    weights = np.array([total / (NUM_LABELS * counts[i]) for i in range(NUM_LABELS)])
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)

class_weights = compute_class_weights(train_samples)
print("Poids de classe :", {ID2LABEL[i]: f"{w:.3f}" for i, w in enumerate(class_weights.tolist())})

Poids de classe : {'premise': '1.226', 'conclusion': '4.934', 'none': '0.505'}


## 3. Dataset PyTorch et tokenisation

La classe EnthymemeDataset prélève chaque tweet, le tokenise à l'aide du tokenizer de DeBERTa et le ramène à une longueur fixe de 128 tokens (ce qui est suffisant pour un tweet). Chaque lot renvoie input_ids, attention_mask, labelset soft_labels.

In [7]:
class EnthymemeDataset(Dataset):
    """
    Dataset PyTorch pour la tâche de classification d'enthymèmes.

    Chaque exemple retourne :
      - input_ids, attention_mask, token_type_ids : tokens du tweet
      - labels : indice du label majoritaire (hard label)
      - soft_labels : distribution des votes annotateurs (pour évaluation)
    """

    def __init__(self, samples, tokenizer):
        self.samples   = samples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Tokenisation du texte brut du tweet
        encoding = self.tokenizer(
            sample["text"],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        item = {
            "input_ids"      : encoding["input_ids"].squeeze(0),
            "attention_mask" : encoding["attention_mask"].squeeze(0),
            "labels"         : torch.tensor(sample["hard_label"], dtype=torch.long),
            "soft_labels"    : torch.tensor(sample["soft_label"],  dtype=torch.float),
        }

        # token_type_ids : présent pour certains modèles (BERT, DeBERTa), absent pour RoBERTa
        if "token_type_ids" in encoding:
            item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)

        return item


# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = EnthymemeDataset(train_samples, tokenizer)
dev_dataset   = EnthymemeDataset(dev_samples,   tokenizer)

print(f"Exemple tokenisé — clés : {list(train_dataset[0].keys())}")

/data-lachesis/common/propp/.venv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Exemple tokenisé — clés : ['input_ids', 'attention_mask', 'labels', 'soft_labels', 'token_type_ids']


## 4. Métriques d'évaluation

Deux métriques calculées à chaque itération :
-  F1 macro 3 classes : évalue la distinction entre prémisse / conclusion / aucune
-  F1 macro 2 classes (officielle) : regroupe prémisse+conclusion en « implicite » vs aucune — c'est la métrique de classement officielle de la shared task.

La fonction evaluate_with_soft_labels calcule quant à elle l'entropie croisée entre les probabilités prédites et les soft labels — requise pour la soumission mais non utilisée pour choisir le meilleur modèle.

In [8]:
def compute_metrics(eval_pred):
    """
    Fonction de métriques passée au HuggingFace Trainer.

    Calcule :
    - f1_macro_3class : F1 macro sur les 3 labels originaux
    - f1_macro_2class : F1 macro en mode fusionné (premise+conclusion → 'implicit')
                        → MÉTRIQUE OFFICIELLE DE CLASSEMENT
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # ── F1 macro 3 classes ────────────────────────────────────────────────────
    f1_3 = f1_score(labels, preds, average="macro", zero_division=0)

    # ── F1 macro 2 classes fusionnées ─────────────────────────────────────────
    # On fusionne premise (0) et conclusion (1) → classe 'implicit' (0)
    # none (2) → classe 'none' (1)
    def merge(x):
        return 0 if x in [0, 1] else 1

    preds_2  = np.vectorize(merge)(preds)
    labels_2 = np.vectorize(merge)(labels)
    f1_2 = f1_score(labels_2, preds_2, average="macro", zero_division=0)

    return {
        "f1_macro_3class" : round(f1_3, 4),
        "f1_macro_2class" : round(f1_2, 4),  # ← métrique officielle
    }


def evaluate_with_soft_labels(model, dataset, device=DEVICE):
    """
    Évaluation complémentaire : calcule la cross-entropy entre les probabilités
    prédites et les soft labels (distribution des votes annotateurs).
    Métrique demandée pour la soumission officielle (en plus du hard F1).
    """
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)

    all_probs, all_soft = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            soft_labels    = batch["soft_labels"]

            kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
            if "token_type_ids" in batch:
                kwargs["token_type_ids"] = batch["token_type_ids"].to(device)

            outputs = model(**kwargs)
            probs   = F.softmax(outputs.logits, dim=-1).cpu()

            all_probs.append(probs)
            all_soft.append(soft_labels)

    all_probs = torch.cat(all_probs, dim=0)  # [N, 3]
    all_soft  = torch.cat(all_soft,  dim=0)  # [N, 3]

    # Cross-entropy entre prédiction et soft label
    # CE = -Σ soft_label * log(prob)
    ce_loss = -(all_soft * torch.log(all_probs.clamp(min=1e-8))).sum(dim=-1).mean()
    print(f"Cross-entropy sur soft labels (dev) : {ce_loss.item():.4f}")
    return all_probs.numpy()

## 5A. Approche 1 — Full Fine-Tuning

Charge DeBERTa-v3-base avec un classificateur à 3 classes. La classe WeightedCETrainer remplace compute_loss pour utiliser une entropie croisée pondérée — ce qui est essentiel car none représente 66 % de l'ensemble de données et, sans pondération, le modèle aurait tendance à prédire presque toujours cette classe. Les poids sont calculés par fréquence inverse : none a un faible poids, conclusion (seulement 7 %) a un poids très élevé.

Le trainer utilise l'Early Stopping avec une patience de 3 : si le F1 à 2 classes ne s'améliore pas pendant 3 époques consécutives, il s'arrête et charge le meilleur checkpoint.

In [9]:
# ── Chargement du modèle pour le Full Fine-Tuning ─────────────────────────────
# Tous les paramètres seront mis à jour pendant l'entraînement
model_fft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

total_params_fft = sum(p.numel() for p in model_fft.parameters())
print(f"Paramètres entraînables (FFT) : {total_params_fft:,}")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Paramètres entraînables (FFT) : 184,424,451


In [10]:
class WeightedCETrainer(Trainer):
    """
    Sous-classe de Trainer qui remplace la Cross-Entropy standard
    par une version pondérée selon la fréquence inverse des classes.
    Indispensable ici car 'none' représente ~66% du dataset.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        # Les soft_labels ne sont pas utilisés à l'entraînement en Constrained Run 1
        inputs.pop("soft_labels", None)

        outputs = model(**inputs)
        logits  = outputs.logits

        # Pondération par fréquence inverse → pénalise davantage les erreurs
        # sur les classes minoritaires (conclusion en particulier)
        loss = F.cross_entropy(logits, labels, weight=class_weights)

        return (loss, outputs) if return_outputs else loss


# ── Configuration de l'entraînement ──────────────────────────────────────────
training_args_fft = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR / "fft"),
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,         # LR standard pour FFT sur encodeurs
    weight_decay                = 0.01,          # régularisation L2
    warmup_ratio                = 0.1,           # 10% des steps en warmup linéaire
    lr_scheduler_type           = "linear",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro_2class",  # métrique officielle
    greater_is_better           = True,
    logging_steps               = 20,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),  # mixed precision si GPU
    report_to                   = "none",
)

trainer_fft = WeightedCETrainer(
    model           = model_fft,
    args            = training_args_fft,
    train_dataset   = train_dataset,
    eval_dataset    = dev_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Démarrage de l'entraînement — Full Fine-Tuning...")
trainer_fft.train()

Démarrage de l'entraînement — Full Fine-Tuning...


Epoch,Training Loss,Validation Loss,F1 Macro 3class,F1 Macro 2class
1,1.103500,1.099219,0.258800,0.388100
2,1.095700,1.096093,0.291900,0.484800
3,1.067600,1.072257,0.460200,0.641000
4,0.860500,1.017383,0.450300,0.633600
5,0.640400,1.402472,0.465500,0.655900
6,0.355300,1.890550,0.453300,0.675400
7,0.228000,1.946721,0.461400,0.654000
8,0.121700,2.396201,0.406200,0.613000
9,0.086600,2.355536,0.428100,0.584200


TrainOutput(global_step=342, training_loss=0.613411823037075, metrics={'train_runtime': 144.5111, 'train_samples_per_second': 82.969, 'train_steps_per_second': 2.63, 'total_flos': 709826952257280.0, 'train_loss': 0.613411823037075, 'epoch': 9.0})

In [11]:
# ── Évaluation finale du modèle FFT ──────────────────────────────────────────
print("=== Résultats Full Fine-Tuning ===")
results_fft = trainer_fft.evaluate()
print(results_fft)

# Rapport de classification détaillé (précision, rappel, F1 par classe)
preds_fft = trainer_fft.predict(dev_dataset)
pred_labels_fft = np.argmax(preds_fft.predictions, axis=-1)
print("\nRapport de classification (FFT) :")
print(classification_report(
    preds_fft.label_ids,
    pred_labels_fft,
    target_names=list(LABEL2ID.keys())
))

# Cross-entropy sur soft labels
probs_fft = evaluate_with_soft_labels(model_fft.to(DEVICE), dev_dataset)

=== Résultats Full Fine-Tuning ===


{'eval_loss': 1.8905502557754517, 'eval_f1_macro_3class': 0.4533, 'eval_f1_macro_2class': 0.6754, 'eval_runtime': 0.4403, 'eval_samples_per_second': 304.341, 'eval_steps_per_second': 6.814, 'epoch': 9.0}

Rapport de classification (FFT) :
              precision    recall  f1-score   support

     premise       0.56      0.63      0.59        38
  conclusion       0.00      0.00      0.00        11
        none       0.76      0.78      0.77        85

    accuracy                           0.67       134
   macro avg       0.44      0.47      0.45       134
weighted avg       0.64      0.67      0.65       134

Cross-entropy sur soft labels (dev) : 1.0232


## 5B. Approche 2 — LoRA (Parameter-Efficient Fine-Tuning)

Il part du même modèle de base, mais gèle presque tous les paramètres et ajoute de petites matrices ΔW = A·B uniquement dans les projections « query » et « value » de l'attention. Concrètement, au lieu de mettre à jour environ 183 millions de paramètres, il n'en met à jour qu'environ 1 à 2 millions (moins de 1 %). Les différences par rapport au FFT sont les suivantes :
- Taux d'apprentissage plus élevé (5e-4 contre 2e-5) — cela est possible car seuls quelques paramètres sont modifiés
- Planificateur cosinus au lieu de linéaire
- Arrêt précoce avec une patience de 4 au lieu de 3

In [12]:
# ── Chargement du modèle de base + application de LoRA ───────────────────────
base_model_lora = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

# Configuration LoRA :
#   r          : rang des matrices ΔW = A·B (plus r est grand, plus LoRA est expressif)
#   lora_alpha : facteur de mise à l'échelle (convention : alpha = 2*r)
#   target_modules : couches ciblées — query et value des self-attention
#                    (DeBERTa utilise les noms 'query_proj' et 'value_proj')
#   lora_dropout   : dropout appliqué aux adaptateurs (régularisation)
#   bias           : 'none' = on n'adapte pas les biais
lora_config = LoraConfig(
    r               = 16,
    lora_alpha      = 32,
    target_modules  = ["query_proj", "value_proj"],  # adapter si modèle différent
    lora_dropout    = 0.1,
    bias            = "none",
    task_type       = TaskType.SEQ_CLS,
)

model_lora = get_peft_model(base_model_lora, lora_config)

# Comptage des paramètres entraînables vs total
trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_lora.parameters())
print(f"Paramètres entraînables (LoRA) : {trainable:,} / {total:,}")
print(f"Réduction : {100 * trainable / total:.2f}% des paramètres seulement")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Paramètres entraînables (LoRA) : 592,131 / 185,016,582
Réduction : 0.32% des paramètres seulement


In [13]:
# ── Configuration de l'entraînement LoRA ─────────────────────────────────────
# On peut se permettre un LR plus élevé car seul un sous-ensemble de paramètres
# est mis à jour → moins de risque de détruire les représentations pré-entraînées
training_args_lora = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR / "lora"),
    num_train_epochs            = 15,            # plus d'époques possibles (moins d'overfitting)
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 5e-4,          # LR plus élevé approprié pour LoRA
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = "cosine",      # cosine souvent meilleur pour LoRA
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro_2class",
    greater_is_better           = True,
    logging_steps               = 20,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
)

trainer_lora = WeightedCETrainer(
    model           = model_lora,
    args            = training_args_lora,
    train_dataset   = train_dataset,
    eval_dataset    = dev_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=4)],
)

print("Démarrage de l'entraînement — LoRA...")
trainer_lora.train()

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Démarrage de l'entraînement — LoRA...


Epoch,Training Loss,Validation Loss,F1 Macro 3class,F1 Macro 2class
1,1.113900,1.108804,0.147300,0.267800
2,1.099900,1.084791,0.229600,0.534700
3,1.039600,1.037315,0.387800,0.626900
4,0.956100,1.086972,0.317900,0.578700
5,0.937600,1.083628,0.377700,0.586800
6,0.816000,1.521137,0.424000,0.614600
7,0.735500,1.920645,0.455100,0.633600
8,0.688800,2.207973,0.471200,0.633600
9,0.425800,2.423824,0.480500,0.633600
10,0.401600,2.589397,0.451800,0.640400


TrainOutput(global_step=570, training_loss=0.6425778711051272, metrics={'train_runtime': 185.4733, 'train_samples_per_second': 96.968, 'train_steps_per_second': 3.073, 'total_flos': 1191223718023680.0, 'train_loss': 0.6425778711051272, 'epoch': 15.0})

In [14]:
# ── Évaluation finale du modèle LoRA ─────────────────────────────────────────
print("=== Résultats LoRA ===")
results_lora = trainer_lora.evaluate()
print(results_lora)

preds_lora = trainer_lora.predict(dev_dataset)
pred_labels_lora = np.argmax(preds_lora.predictions, axis=-1)
print("\nRapport de classification (LoRA) :")
print(classification_report(
    preds_lora.label_ids,
    pred_labels_lora,
    target_names=list(LABEL2ID.keys())
))

probs_lora = evaluate_with_soft_labels(model_lora.to(DEVICE), dev_dataset)

=== Résultats LoRA ===


{'eval_loss': 2.7222981452941895, 'eval_f1_macro_3class': 0.4614, 'eval_f1_macro_2class': 0.654, 'eval_runtime': 0.5808, 'eval_samples_per_second': 230.702, 'eval_steps_per_second': 5.165, 'epoch': 15.0}

Rapport de classification (LoRA) :
              precision    recall  f1-score   support

     premise       0.46      0.68      0.55        38
  conclusion       0.17      0.09      0.12        11
        none       0.78      0.66      0.71        85

    accuracy                           0.62       134
   macro avg       0.47      0.48      0.46       134
weighted avg       0.64      0.62      0.62       134

Cross-entropy sur soft labels (dev) : 1.2866


## 6. Recherche d'hyperparamètres sur grille (Grid Search)

Cette section explore systématiquement les combinaisons d'hyperparamètres les plus importantes.
Elle est conçue pour tourner sur serveur : chaque run est indépendant, les résultats sont
sauvegardés au fur et à mesure dans un CSV pour ne rien perdre en cas d'interruption.

**Grille explorée :**
| Hyperparamètre | Valeurs (FFT) | Valeurs (LoRA) |
|---|---|---|
| `learning_rate` | 1e-5, 2e-5, 3e-5 | 1e-4, 5e-4, 1e-3 |
| `batch_size` | 8, 16 | 8, 16 |
| `r` (rang LoRA) | — | 8, 16, 32 |
| `warmup_ratio` | 0.06, 0.1 | 0.06, 0.1 |

Fonctions d'assistance 
(free_gpu_memory, run_single_config) -> Le cœur du réseau. 

run_single_config construit le modèle à partir de zéro, l'entraîne avec les paramètres fournis et renvoie un dictionnaire contenant les métriques. À la fin de chaque exécution, elle appelle free_gpu_memory() pour vider la VRAM — ce qui est essentiel sur les serveurs, où les exécutions s'enchaînent dans la même session Python.

In [ ]:
# à changer après verification de la disponibilité du serveur

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
import itertools
import gc
import copy

# ── Chemin de sauvegarde des résultats de la grille ──────────────────────────
# Les résultats sont écrits ligne par ligne : si le serveur s'arrête en cours
# de route, les runs déjà effectués sont conservés dans ce fichier.
GRID_RESULTS_PATH = OUTPUT_DIR / "grid_search_results.csv"


def free_gpu_memory():
    """Libère la mémoire GPU entre deux runs de la grille."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_single_config(
    approach,
    learning_rate,
    batch_size,
    warmup_ratio,
    lora_r=16,
    num_epochs=10,
    run_id="",
    is_final_run = False,
):
    """
    Entraîne et évalue un modèle pour une configuration d'hyperparamètres donnée.

    Paramètres
    ----------
    approach      : 'fft' ou 'lora'
    learning_rate : taux d'apprentissage
    batch_size    : taille de batch (train)
    warmup_ratio  : fraction des steps consacrée au warmup
    lora_r        : rang des matrices LoRA (ignoré si approach='fft')
    num_epochs    : nombre maximum d'époques (early stopping actif)
    run_id        : identifiant textuel pour les logs

    Retourne
    --------
    dict avec les métriques du meilleur checkpoint sur le dev set
    """

    print(f"\n{'='*60}")
    print(f"Run : {run_id}")
    print(f"  approach={approach}, lr={learning_rate}, bs={batch_size}, "
          f"warmup={warmup_ratio}, lora_r={lora_r}")
    print(f"{'='*60}")

    # ── Construction du modèle ────────────────────────────────────────────────
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    if approach == "lora":
        # Application de LoRA avec le rang spécifié par la grille
        cfg = LoraConfig(
            r              = lora_r,
            lora_alpha     = lora_r * 2,  # convention : alpha = 2 * r
            target_modules = ["query_proj", "value_proj"],
            lora_dropout   = 0.1,
            bias           = "none",
            task_type      = TaskType.SEQ_CLS,
        )
        model = get_peft_model(base_model, cfg)
        # Scheduler cosine plus adapté à LoRA (descente plus douce en fin d'entraînement)
        scheduler = "cosine"
    else:
        model    = base_model
        scheduler = "linear"

    # ── Répertoire de sortie propre à ce run ──────────────────────────────────
    run_output_dir = OUTPUT_DIR / "grid" / run_id
    run_output_dir.mkdir(parents=True, exist_ok=True)

    # ── Arguments d'entraînement ──────────────────────────────────────────────
    args = TrainingArguments(
        output_dir                  = str(run_output_dir),
        num_train_epochs            = num_epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = 32,
        learning_rate               = learning_rate,
        weight_decay                = 0.01,
        warmup_ratio                = warmup_ratio,
        lr_scheduler_type           = scheduler,
        eval_strategy               = "epoch",
        save_strategy          = "epoch" if is_final_run else "no",
        load_best_model_at_end = True    if is_final_run else False,
        metric_for_best_model       = "f1_macro_2class",
        greater_is_better           = True,
        logging_steps               = 50,
        seed                        = SEED,
        fp16                        = torch.cuda.is_available(),
        report_to                   = "none",
        # Désactive la sauvegarde des checkpoints intermédiaires pour économiser
        # l'espace disque sur le serveur (on garde seulement le meilleur)
        save_total_limit       = 1       if is_final_run else 0,
    )

    trainer = WeightedCETrainer(
        model           = model,
        args            = args,
        train_dataset   = train_dataset,
        eval_dataset    = dev_dataset,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()

    # ── Récupération des métriques du meilleur checkpoint ─────────────────────
    eval_results = trainer.evaluate()

    result_row = {
        "run_id"        : run_id,
        "approach"      : approach,
        "learning_rate" : learning_rate,
        "batch_size"    : batch_size,
        "warmup_ratio"  : warmup_ratio,
        "lora_r"        : lora_r if approach == "lora" else "N/A",
        "f1_2class"     : eval_results.get("eval_f1_macro_2class", -1),
        "f1_3class"     : eval_results.get("eval_f1_macro_3class", -1),
        "best_epoch"    : trainer.state.best_epoch if hasattr(trainer.state, "best_epoch") else -1,
    }

    # ── Libération de la mémoire GPU avant le prochain run ────────────────────
    del model, trainer, base_model
    free_gpu_memory()

    return result_row

Définition des grilles 

Définissez GRID_FFT et GRID_LORA sous forme de dictionnaires. Le produit cartésien est généré automatiquement avec itertools.product. Au total : 12 exécutions FFT + 36 exécutions LoRA = 48 exécutions. Sur un serveur équipé d'un bon GPU (A100/V100), chaque exécution dure environ 5 à 10 minutes, ce qui signifie que la grille entière nécessite environ 4 à 8 heures.

In [21]:
# ── Définition des grilles par approche ──────────────────────────────────────

# Grille pour le Full Fine-Tuning
# 3 lr × 2 batch_size × 2 warmup = 12 configurations
GRID_FFT = {
    "learning_rate" : [1e-5, 2e-5, 3e-5],
    "batch_size"    : [8, 16],
    "warmup_ratio"  : [0.06, 0.1],
    "lora_r"        : [None],  # non utilisé en FFT, placeholder pour uniformité
}

# Grille pour LoRA
# 3 lr × 2 batch_size × 2 warmup × 3 r = 36 configurations
# Sur serveur avec une bonne GPU, chaque run prend ~5-10 min → ~3-6h au total
GRID_LORA = {
    "learning_rate" : [1e-4, 5e-4, 1e-3],
    "batch_size"    : [8, 16],
    "warmup_ratio"  : [0.06, 0.1],
    "lora_r"        : [8, 16, 32],
}


def build_run_list(approach, grid):
    """
    Génère la liste de toutes les configurations à tester pour une approche donnée.
    Chaque configuration est un dict prêt à être passé à run_single_config().
    """
    keys   = list(grid.keys())
    values = list(grid.values())
    runs   = []
    for combo in itertools.product(*values):
        config = dict(zip(keys, combo))
        # Construction d'un identifiant lisible pour les logs et le système de fichiers
        if approach == "fft":
            run_id = (f"fft_lr{config['learning_rate']:.0e}"
                      f"_bs{config['batch_size']}"
                      f"_w{config['warmup_ratio']}")
        else:
            run_id = (f"lora_lr{config['learning_rate']:.0e}"
                      f"_bs{config['batch_size']}"
                      f"_w{config['warmup_ratio']}"
                      f"_r{config['lora_r']}")
        config["approach"] = approach
        config["run_id"]   = run_id
        runs.append(config)
    return runs


run_list_fft  = build_run_list("fft",  GRID_FFT)
run_list_lora = build_run_list("lora", GRID_LORA)
all_runs      = run_list_fft + run_list_lora

print(f"Configurations FFT  : {len(run_list_fft)}")
print(f"Configurations LoRA : {len(run_list_lora)}")
print(f"Total               : {len(all_runs)} runs")

Configurations FFT  : 12
Configurations LoRA : 36
Total               : 48 runs


Exécution avec reprise automatique 

C'est l'aspect le plus important dans le contexte d'un serveur. Chaque exécution est immédiatement enregistrée dans le fichier outputs/grid_search_results.csv. Si le serveur s'arrête, lors de la prochaine exécution, le notebook détecte les exécutions déjà terminées et reprend à partir de la première manquante — rien n'est perdu. Les erreurs OOM (out of memory) sont interceptées sans provoquer le plantage de l'ensemble de la grille.

In [ ]:
# ── Exécution de la grille avec reprise automatique ──────────────────────────
# Si le fichier de résultats existe déjà (run interrompu sur le serveur),
# on recharge les runs déjà effectués et on reprend à partir du premier manquant.

if GRID_RESULTS_PATH.exists():
    df_results = pd.read_csv(GRID_RESULTS_PATH)
    completed_ids = set(df_results["run_id"].tolist())
    print(f"Reprise détectée : {len(completed_ids)} runs déjà effectués.")
else:
    df_results    = pd.DataFrame()
    completed_ids = set()
    print("Démarrage d'une nouvelle grille de recherche.")

# Filtrage des runs restants
remaining_runs = [r for r in all_runs if r["run_id"] not in completed_ids]
print(f"Runs restants : {len(remaining_runs)} / {len(all_runs)}")

# ── Boucle principale ─────────────────────────────────────────────────────────
new_rows = []
for i, config in enumerate(remaining_runs):
    print(f"\nProgression : {i+1}/{len(remaining_runs)} runs restants")

    try:
        row = run_single_config(
            approach      = config["approach"],
            learning_rate = config["learning_rate"],
            batch_size    = config["batch_size"],
            warmup_ratio  = config["warmup_ratio"],
            lora_r        = config["lora_r"] if config["lora_r"] is not None else 16,
            run_id        = config["run_id"],
        )
        new_rows.append(row)

        # Sauvegarde incrémentale : on écrit après chaque run terminé
        # → si le serveur s'arrête, on ne perd que le run en cours
        df_new        = pd.DataFrame(new_rows)
        df_results    = pd.concat([df_results, df_new], ignore_index=True)
        df_results.to_csv(GRID_RESULTS_PATH, index=False)
        new_rows      = []  # reset du buffer après sauvegarde

        print(f"  → f1_2class={row['f1_2class']:.4f}  f1_3class={row['f1_3class']:.4f}  "
              f"[sauvegardé dans {GRID_RESULTS_PATH}]")

    except Exception as e:
        # On ne fait pas planter toute la grille pour un run défaillant
        # (ex. OOM sur une configuration avec batch_size=16 et modèle large)
        print(f"  ERREUR sur le run {config['run_id']} : {e}")
        print("  Run ignoré, passage au suivant.")
        free_gpu_memory()
        continue

print("\nGrille de recherche terminée.")
print(f"Résultats complets sauvegardés dans : {GRID_RESULTS_PATH}")

Analyse et réentraînement final

Trois analyses automatiques : les 10 meilleures configurations, la meilleure pour l'approche et une analyse de sensibilité (impact moyen de chaque hyperparamètre). Puis un réentraînement sur 20 époques avec la meilleure configuration trouvée.

In [23]:
# ── Analyse des résultats de la grille ───────────────────────────────────────

df_results = pd.read_csv(GRID_RESULTS_PATH)

# Tri par F1 2-class décroissant (métrique officielle)
df_sorted = df_results.sort_values("f1_2class", ascending=False)

print("=== TOP 10 configurations (par F1 macro 2-class) ===")
print(df_sorted[["run_id", "approach", "f1_2class", "f1_3class"]].head(10).to_string(index=False))

# Meilleure configuration par approche
print("\n=== Meilleure configuration par approche ===")
for approach in ["fft", "lora"]:
    subset = df_sorted[df_sorted["approach"] == approach]
    if len(subset) == 0:
        continue
    best = subset.iloc[0]
    print(f"\n[{approach.upper()}]")
    print(f"  run_id        : {best['run_id']}")
    print(f"  learning_rate : {best['learning_rate']}")
    print(f"  batch_size    : {best['batch_size']}")
    print(f"  warmup_ratio  : {best['warmup_ratio']}")
    if approach == "lora":
        print(f"  lora_r        : {best['lora_r']}")
    print(f"  F1 2-class    : {best['f1_2class']:.4f}  (métrique officielle)")
    print(f"  F1 3-class    : {best['f1_3class']:.4f}")

# Analyse de sensibilité : impact moyen de chaque hyperparamètre
print("\n=== Sensibilité aux hyperparamètres (F1 2-class moyen par valeur) ===")
for col in ["learning_rate", "batch_size", "warmup_ratio"]:
    print(f"\n  {col} :")
    print(df_results.groupby(col)["f1_2class"].mean().sort_values(ascending=False).to_string())

# Analyse spécifique au rang LoRA
df_lora_only = df_results[df_results["approach"] == "lora"]
if len(df_lora_only) > 0:
    print("\n  lora_r (LoRA uniquement) :")
    print(df_lora_only.groupby("lora_r")["f1_2class"].mean().sort_values(ascending=False).to_string())

=== TOP 10 configurations (par F1 macro 2-class) ===
                    run_id approach  f1_2class  f1_3class
     fft_lr1e-05_bs8_w0.06      fft     0.7104     0.5260
lora_lr1e-04_bs16_w0.1_r32     lora     0.6794     0.4365
 lora_lr5e-04_bs16_w0.1_r8     lora     0.6739     0.4730
  lora_lr1e-04_bs8_w0.1_r8     lora     0.6694     0.4470
lora_lr1e-03_bs16_w0.1_r16     lora     0.6656     0.4651
 lora_lr5e-04_bs8_w0.1_r16     lora     0.6573     0.5220
 lora_lr1e-04_bs8_w0.06_r8     lora     0.6548     0.4372
 lora_lr1e-04_bs8_w0.1_r16     lora     0.6500     0.4626
      fft_lr3e-05_bs8_w0.1      fft     0.6461     0.4682
lora_lr1e-04_bs8_w0.06_r32     lora     0.6448     0.4466

=== Meilleure configuration par approche ===

[FFT]
  run_id        : fft_lr1e-05_bs8_w0.06
  learning_rate : 1e-05
  batch_size    : 8
  warmup_ratio  : 0.06
  F1 2-class    : 0.7104  (métrique officielle)
  F1 3-class    : 0.5260

[LORA]
  run_id        : lora_lr1e-04_bs16_w0.1_r32
  learning_rate : 0.000

In [30]:
# ── Ré-entraînement final avec la meilleure configuration ────────────────────
# Une fois la grille analysée, on relance un entraînement complet avec
# la meilleure configuration identifiée, en utilisant plus d'époques
# (l'early stopping s'en chargera si on surapprenait avant).

# Sélectionner manuellement 'fft' ou 'lora' selon les résultats ci-dessus
BEST_APPROACH = "fft"  # ← à modifier selon les résultats de la grille

best_config = (
    df_results[df_results["approach"] == BEST_APPROACH]
    .sort_values("f1_2class", ascending=False)
    .iloc[0]
)

print(f"Configuration finale sélectionnée ({BEST_APPROACH.upper()}) :")
print(best_config.to_string())

# Ré-entraînement avec un budget d'époques plus généreux
best_model_row = run_single_config(
    approach      = BEST_APPROACH,
    learning_rate = float(best_config["learning_rate"]),
    batch_size    = int(best_config["batch_size"]),
    warmup_ratio  = float(best_config["warmup_ratio"]),
    lora_r        = int(best_config["lora_r"]) if BEST_APPROACH == "lora" else 16,
    num_epochs    = 20,   # budget plus large ; l'early stopping arrêtera au bon moment
    run_id        = f"best_{BEST_APPROACH}_final",
)

print(f"\nF1 final (meilleure config) : {best_model_row['f1_2class']:.4f}")

Configuration finale sélectionnée (FFT) :
run_id           fft_lr1e-05_bs8_w0.06
approach                           fft
learning_rate                  0.00001
batch_size                           8
warmup_ratio                      0.06
lora_r                             NaN
f1_2class                       0.7104
f1_3class                        0.526
best_epoch                          -1

Run : best_fft_final
  approach=fft, lr=1e-05, bs=8, warmup=0.06, lora_r=16


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,F1 Macro 3class,F1 Macro 2class
1,1.095400,1.099104,0.288200,0.468000
2,1.087700,1.086459,0.291800,0.472100
3,1.036500,1.104010,0.417700,0.639500
4,0.859800,1.016934,0.485100,0.622700
5,0.625500,1.094620,0.478100,0.614600
6,0.370900,1.518480,0.421800,0.612300



F1 final (meilleure config) : 0.6123


## 7. Comparaison des deux approches

Tableau comparatif des deux approches et génération du fichier CSV de soumission avec hard_label + vecteur de probabilité pour chaque tweet.

In [31]:
# ── Tableau récapitulatif des performances ────────────────────────────────────
comparison = pd.DataFrame({
    "Approche"         : ["Full Fine-Tuning", "LoRA"],
    "F1 macro 3-class" : [
        results_fft.get("eval_f1_macro_3class", "N/A"),
        results_lora.get("eval_f1_macro_3class", "N/A"),
    ],
    "F1 macro 2-class (officiel)" : [
        results_fft.get("eval_f1_macro_2class", "N/A"),
        results_lora.get("eval_f1_macro_2class", "N/A"),
    ],
})
print(comparison.to_string(index=False))

        Approche  F1 macro 3-class  F1 macro 2-class (officiel)
Full Fine-Tuning            0.4533                       0.6754
            LoRA            0.4614                       0.6540


## 7.1 Extension Grid Search

In [ ]:
# ── Grille étendue FFT uniquement — zone lr < 1e-5 ───────────────────────────
# La grille initiale montre que lr=1e-5 est optimal avec bs=8 et warmup=0.06.
# On explore ici la zone inféreure pour vérifier si le F1 continue à progresser.

GRID_FFT_EXTENDED = {
    "learning_rate" : [3e-6, 5e-6, 8e-6],
    "batch_size"    : [8],
    "warmup_ratio"  : [0.06],
    "lora_r"        : [None],
}

def build_run_list(approach, grid):
    """
    Génère la liste de toutes les configurations à tester pour une approche donnée.
    Chaque configuration est un dict prêt à être passé à run_single_config().
    """
    keys   = list(grid.keys())
    values = list(grid.values())
    runs   = []
    for combo in itertools.product(*values):
        config = dict(zip(keys, combo))
        # Construction d'un identifiant lisible pour les logs et le système de fichiers
        if approach == "fft":
            run_id = (f"fft_lr{config['learning_rate']:.0e}"
                      f"_bs{config['batch_size']}"
                      f"_w{config['warmup_ratio']}")
        else:
            run_id = (f"lora_lr{config['learning_rate']:.0e}"
                      f"_bs{config['batch_size']}"
                      f"_w{config['warmup_ratio']}"
                      f"_r{config['lora_r']}")
        config["approach"] = approach
        config["run_id"]   = run_id
        runs.append(config)
    return runs

run_list_extended = build_run_list("fft", GRID_FFT_EXTENDED)

print(f"Configurations à tester : {len(run_list_extended)}")
for r in run_list_extended:
    print(f"  {r['run_id']}")

In [ ]:
# ── Exécution de la grille étendue avec reprise automatique ──────────────────
# Les résultats sont ajoutés au même fichier CSV que la grille initiale
# pour permettre une analyse comparative unifiée.

if GRID_RESULTS_PATH.exists():
    df_results = pd.read_csv(GRID_RESULTS_PATH)
    completed_ids = set(df_results["run_id"].tolist())
    print(f"Résultats existants chargés : {len(df_results)} runs déjà effectués.")
else:
    df_results    = pd.DataFrame()
    completed_ids = set()

remaining_runs = [r for r in run_list_extended if r["run_id"] not in completed_ids]
print(f"Runs restants : {len(remaining_runs)} / {len(run_list_extended)}")

new_rows = []
for i, config in enumerate(remaining_runs):
    print(f"\nProgression : {i+1}/{len(remaining_runs)}")
    try:
        row = run_single_config(
            approach      = config["approach"],
            learning_rate = config["learning_rate"],
            batch_size    = config["batch_size"],
            warmup_ratio  = config["warmup_ratio"],
            lora_r        = 16,  # non utilisé en FFT, valeur placeholder
            run_id        = config["run_id"],
        )
        new_rows.append(row)

        df_new     = pd.DataFrame(new_rows)
        df_results = pd.concat([df_results, df_new], ignore_index=True)
        df_results.to_csv(GRID_RESULTS_PATH, index=False)
        new_rows   = []

        print(f"  → f1_2class={row['f1_2class']:.4f}  f1_3class={row['f1_3class']:.4f}  "
              f"[sauvegardé]")

    except Exception as e:
        print(f"  ERREUR sur {config['run_id']} : {e}")
        free_gpu_memory()
        continue

print("\nGrille étendue terminée.")

In [ ]:
# ── Analyse comparative : grille initiale + grille étendue ───────────────────
# On relit le CSV complet pour avoir une vue unifiée des deux grilles.

df_results = pd.read_csv(GRID_RESULTS_PATH)
df_sorted  = df_results.sort_values("f1_2class", ascending=False)

print("=== TOP 10 configurations (grilles initiale + étendue) ===")
print(df_sorted[["run_id", "approach", "f1_2class", "f1_3class"]].head(10).to_string(index=False))

# Focus sur les runs FFT pour visualiser la tendance selon le learning rate
df_fft_all = df_sorted[df_sorted["approach"] == "fft"].copy()
print("\n=== Tendance FFT par learning rate (toutes grilles) ===")
print(df_fft_all[["run_id", "learning_rate", "f1_2class", "f1_3class"]]
      .sort_values("learning_rate")
      .to_string(index=False))

# ── Sélection de la meilleure configuration pour le re-training final ─────────
best = df_sorted.iloc[0]
print(f"\n=== Meilleure configuration globale ===")
print(f"  run_id        : {best['run_id']}")
print(f"  learning_rate : {best['learning_rate']}")
print(f"  batch_size    : {best['batch_size']}")
print(f"  warmup_ratio  : {best['warmup_ratio']}")
print(f"  F1 2-class    : {best['f1_2class']:.4f}")
print(f"  F1 3-class    : {best['f1_3class']:.4f}")

# ── Re-training final avec la meilleure configuration ────────────────────────
best_model_row = run_single_config(
    approach      = str(best["approach"]),
    learning_rate = float(best["learning_rate"]),
    batch_size    = int(best["batch_size"]),
    warmup_ratio  = float(best["warmup_ratio"]),
    lora_r        = int(best["lora_r"]) if str(best["approach"]) == "lora" else 16,
    num_epochs    = 20,
    run_id        = "best_final",
    is_final_run  = True,   # ← attiva il salvataggio del miglior checkpoint
)

print(f"\nF1 final (meilleure config) : {best_model_row['f1_2class']:.4f}")

## 8. Génération des fichiers de soumission

In [32]:
# ── Chargement du jeu de test (sans annotations) ─────────────────────────────
# Le test set ne contient pas de champ 'annotations' ni 'majority_label' :
# on charge uniquement le texte et l'id de chaque tweet.

TEST_FILE = Path("/data-lachesis/common/propp/DeniseAtzori/notebooks/DeniseAtzori/LoRA/test_v2.json")  # ← inserisci il path qui

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_raw = json.load(f)

# Construction de la liste de samples (sans hard_label ni soft_label)
test_samples = [
    {
        "id"   : entry["id"],
        "text" : entry["tweet_text"],
        # Placeholders nécessaires pour que EnthymemeDataset fonctionne
        "hard_label" : 0,
        "soft_label" : [1/3, 1/3, 1/3],
    }
    for entry in test_raw
]

# Tokenisation
test_dataset = EnthymemeDataset(test_samples, tokenizer)

print(f"Test set chargé : {len(test_samples)} tweets")

Test set chargé : 148 tweets


In [33]:
def generate_submission(model, dataset, samples, run_name, device=DEVICE):
    """
    Génère le fichier de soumission officiel au format demandé.

    Chaque ligne contient :
      - id           : identifiant du tweet
      - hard_label   : label prédit (premise / conclusion / none)
      - prob_premise : probabilité de la classe 'premise'
      - prob_conc    : probabilité de la classe 'conclusion'
      - prob_none    : probabilité de la classe 'none'
    """
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)

    all_probs, all_preds = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
            if "token_type_ids" in batch:
                kwargs["token_type_ids"] = batch["token_type_ids"].to(device)

            outputs = model(**kwargs)
            probs   = F.softmax(outputs.logits, dim=-1).cpu().numpy()
            preds   = probs.argmax(axis=-1)

            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())

    # Construction du DataFrame de soumission
    rows = []
    for i, sample in enumerate(samples):
        rows.append({
            "id"          : sample["id"],
            "hard_label"  : ID2LABEL[all_preds[i]],
            "prob_premise" : round(all_probs[i][0], 4),
            "prob_conclusion": round(all_probs[i][1], 4),
            "prob_none"   : round(all_probs[i][2], 4),
        })

    df = pd.DataFrame(rows)
    out_path = OUTPUT_DIR / f"submission_run1_{run_name}.csv"
    df.to_csv(out_path, index=False)
    print(f"Fichier de soumission sauvegardé : {out_path}")
    return df


# Soumission sur le dev set (à remplacer par le test set quand disponible)
# Le test set sera fourni sans annotations — même pipeline, fichier JSON différent
df_fft  = generate_submission(model_fft.to(DEVICE),  test_dataset, test_samples, "fft")
df_lora = generate_submission(model_lora.to(DEVICE), test_dataset, test_samples, "lora")

print("\nAperçu de la soumission FFT :")
print(df_fft.head())

Fichier de soumission sauvegardé : outputs/submission_run1_fft.csv
Fichier de soumission sauvegardé : outputs/submission_run1_lora.csv

Aperçu de la soumission FFT :
   id hard_label  prob_premise  prob_conclusion  prob_none
0   4       none        0.3820           0.0156     0.6024
1  10       none        0.0380           0.0075     0.9545
2  18       none        0.1684           0.0158     0.8158
3  36       none        0.0276           0.0045     0.9679
4  39       none        0.3238           0.0105     0.6658


## 9. Notes méthodologiques

### Choix du modèle de base
- **DeBERTa-v3-base** : disentangled attention (position + contenu séparés), très performant sur les tâches de classification de texte court. Recommandé en priorité.
- **RoBERTa-large** : alternative, un peu plus lent mais parfois plus stable en FFT.

### Hyperparamètres à faire varier (grille de recherche suggérée)
| Paramètre | Valeurs à tester |
|-----------|------------------|
| `learning_rate` (FFT) | 1e-5, 2e-5, 3e-5 |
| `learning_rate` (LoRA) | 1e-4, 5e-4, 1e-3 |
| `r` (LoRA) | 8, 16, 32 |
| `batch_size` | 8, 16 |

### Gestion du déséquilibre
Le label `conclusion` est très minoritaire (~7%). En plus de la pondération de la loss,
on peut envisager du sur-échantillonnage (SMOTE textuel) ou de l'augmentation de données
pour cette classe si les performances restent insuffisantes.

### Passage au test set
Quand le test set sera disponible, charger le fichier JSON correspondant et appeler
`generate_submission()` avec le modèle sélectionné sur le dev set.